# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata information
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")
print(f"Dataset identifier: {metadata['identifier']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

All entities (record sets, fields, columns) are referenced by their `@id`.

In [ ]:
# List all available record sets and their @ids
record_sets = dataset.record_sets
print("Record sets found:")
for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']} | Name: {rs.get('name', 'N/A')}")

# List fields for each record set
for rs in record_sets:
    fields = rs.get('field', [])
    print(f"\nFields for RecordSet @id {rs['@id']}:")
    for f in fields:
        if isinstance(f, dict):
            print(f"  - Field @id: {f['@id']} | Name: {f.get('name', 'N/A')}")
        else:
            print(f"  - Field @id: {f}")

# Show a preview of records from each record set
for rs in record_sets:
    recset_id = rs['@id']
    print(f"\nSample records from RecordSet @id {recset_id}:")
    records = list(dataset.records(record_set=recset_id))
    for rec in records[:2]:  # Preview first two records
        print(rec)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Get the @ids for all record sets
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

# Extract records into DataFrames for each record set
for recset_id in record_set_ids:
    records = list(dataset.records(record_set=recset_id))
    if records:
        dataframes[recset_id] = pd.DataFrame(records)

# Print the columns for each loaded DataFrame
for recset_id, df in dataframes.items():
    print(f"\nColumns for RecordSet @id {recset_id}:")
    print(df.columns.tolist())
    print(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

All processing uses fields referenced by their `@id`.

In [ ]:
# Select a record set and numeric field by @id for EDA
# Replace example @ids with those found in the overview above.
example_record_set_id = record_set_ids[0] if record_set_ids else None
df = dataframes.get(example_record_set_id)

if df is not None:
    # Find a numeric column
    numeric_columns = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
    if numeric_columns:
        numeric_field_id = numeric_columns[0]
        threshold = 10

        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with '{numeric_field_id}' > {threshold}:")
        print(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized '{numeric_field_id}' for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Grouping: Select a categorical column
        categorical_columns = df.select_dtypes(include=['object']).columns.tolist()
        group_field_id = categorical_columns[0] if categorical_columns else None
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean of '{numeric_field_id}' by '{group_field_id}':")
            print(grouped_df.head())
    else:
        print("No numeric columns found for EDA in this record set.")
else:
    print("No DataFrame found for the specified record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.
All visualizations reference fields by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram for the numeric field
if df is not None and numeric_columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id], bins=15, kde=True)
    plt.title(f"Distribution of '{numeric_field_id}' in RecordSet {example_record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Plot boxplot grouped by categorical field
    if group_field_id:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"'{numeric_field_id}' by '{group_field_id}'")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Explored dataset structure using Croissant schema and `mlcroissant`.
- Identified available record sets and fields via `@id`.
- Loaded tabular data into Pandas DataFrames for analysis.
- Applied filtering, normalization, and grouping to numeric and categorical fields by their `@id`.
- Visualized distributions and relationships in the data.

This approach demonstrates FAIR-compatible exploration and processing of tabular data using Croissant and the mlcroissant Python library.